# 02 — Linear systems and numerical diagnostics

**Goal:** solve linear systems, interpret residuals, explore singular matrices and pseudoinverses, and connect eigenvalues with matrix behavior.

The library delegates its numerical computations to NumPy. Use this notebook to understand inputs, outputs, and diagnostics rather than assuming every floating-point answer is exact.

In [ ]:
# Locate the repository when running locally, in Jupyter, or in Google Colab.
from pathlib import Path
import subprocess
import sys

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "numpy-matrix-operations"]
repo_root = next((p for p in candidates if (p / "src" / "matrixlib").is_dir()), None)

if repo_root is None:
    clone_path = Path.cwd() / "numpy-matrix-operations"
    subprocess.run(
        ["git", "clone", "--branch", "docs/notebooks-complete-audit",
         "https://github.com/souma888/numpy-matrix-operations.git", str(clone_path)],
        check=True,
    )
    repo_root = clone_path

sys.path.insert(0, str(repo_root / "src"))
print("Using repository:", repo_root.resolve())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matrixlib import Matrix

np.set_printoptions(precision=6, suppress=True)

## 1. Solve Ax = b

For a square nonsingular matrix A, `solve(b)` computes x satisfying Ax = b. Prefer a solver over explicitly computing A inverse and multiplying by b.

In [ ]:
A = Matrix([[3, 1], [1, 2]])
b = np.array([9.0, 8.0])
x = A.solve(b)
residual = A.data @ x - b

print("solution x:", x)
print("A @ x:", A.data @ x)
print("b:", b)
print("residual r = Ax - b:", residual)
print("residual norm:", np.linalg.norm(residual))
print("condition number:", A.condition_number())

## 2. Residual versus sensitivity

The residual measures how closely the computed solution satisfies the equations. The condition number measures sensitivity to perturbations. A small residual does not guarantee that the answer changes little when inputs change.

The following example compares a well-conditioned matrix with an ill-conditioned diagonal matrix.

In [ ]:
well = np.diag([1.0, 2.0])
ill = np.diag([1.0, 1e-10])
for name, array in [("well-conditioned", well), ("ill-conditioned", ill)]:
    m = Matrix(array)
    rhs = np.array([1.0, 1.0])
    solution = m.solve(rhs)
    print(name)
    print("  condition number:", m.condition_number())
    print("  solution:", solution)
    print("  residual norm:", np.linalg.norm(array @ solution - rhs))

## 3. Perturb the right-hand side

Small changes in b can produce large changes in x when A is ill-conditioned. This experiment makes the perturbation explicit.

In [ ]:
A = np.diag([1.0, 1e-8])
b = np.array([1.0, 1.0])
delta = np.array([0.0, 1e-8])

x = np.linalg.solve(A, b)
x_perturbed = np.linalg.solve(A, b + delta)

print("condition number:", np.linalg.cond(A))
print("original x:", x)
print("perturbed x:", x_perturbed)
print("change in b norm:", np.linalg.norm(delta))
print("change in x norm:", np.linalg.norm(x_perturbed - x))

## 4. Singular systems and pseudoinverse

A singular matrix does not have a unique inverse. The pseudoinverse can produce a least-squares solution for rectangular or rank-deficient systems. It does not turn an inconsistent system into an exact solution; inspect the residual.

In [ ]:
S = Matrix([[1, 2], [2, 4]])
print("rank:", S.rank())
print("condition number:", S.condition_number())
try:
    S.solve([3, 6])
except ValueError as error:
    print("solve error:", error)

x_pinv = S.pseudoinverse().data @ np.array([3.0, 6.0])
print("pseudoinverse solution:", x_pinv)
print("residual:", S.data @ x_pinv - np.array([3.0, 6.0]))

## 5. Multiple right-hand sides

You can solve AX = B for multiple right-hand-side columns in one call.

In [ ]:
A = Matrix([[3, 1], [1, 2]])
B = np.array([[9.0, 1.0],
              [8.0, 0.0]])
X = A.solve(B)
print("X:\n", X)
print("A @ X:\n", A.data @ X)
print("residual matrix:\n", A.data @ X - B)

## 6. Eigenvalues and eigenvectors

An eigenpair satisfies Av = λv. Eigenvectors are returned as columns of the eigenvector matrix. Eigenvalues can be complex even when the input matrix is real.

In [ ]:
E = Matrix([[2, 1], [0, 3]])
values, vectors = E.eigenvectors()
print("eigenvalues:", values)
print("eigenvectors (columns):\n", vectors)
for i, value in enumerate(values):
    v = vectors[:, i]
    print(f"pair {i}: ||Av - lambda*v|| =", np.linalg.norm(E.data @ v - value * v))

## 7. Plot the singular-value spectrum

Singular values help explain numerical rank and conditioning. A very small minimum singular value can make a matrix ill-conditioned or rank-deficient under a chosen tolerance.

In [ ]:
M = np.array([[1.0, 1.0], [1.0, 1.0 + 1e-5]])
singular_values = np.linalg.svd(M, compute_uv=False)
plt.figure(figsize=(5, 3))
plt.semilogy(range(1, len(singular_values) + 1), singular_values, "o-")
plt.xticks([1, 2])
plt.xlabel("Singular value index")
plt.ylabel("Singular value (log scale)")
plt.title("Singular-value spectrum")
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()
print("singular values:", singular_values)
print("condition number:", np.linalg.cond(M))

## Exercises

1. Solve a 3 × 3 system and verify the residual.
2. Compare solving with `np.linalg.solve(A, b)` against `np.linalg.inv(A) @ b` for a well-conditioned system. Explain why the solver is preferred.
3. Make a rank-deficient rectangular matrix and compute its pseudoinverse.
4. Perturb a matrix entry and measure how much the solution changes.
5. Find a real matrix with complex eigenvalues and verify an eigenpair residual.
6. Explain the difference between residual size and condition number.

**Numerical caution:** condition numbers depend on the norm and matrix; they are indicators of sensitivity, not universal error guarantees.